In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
import os
import pandas as pd
from PI_NN_load_call import PE_call as PE


## Parameter Setup

In [ ]:
# Parameters
d = 1
total_time = 0.5
n_time_steps = 100
dt = total_time / n_time_steps
K = 10.0  # penalty factor
r = 0.05
sigma = 0.4
strike = 40.0
x_0 = 40.0
test_size = 2**20
dividend_train = 0.05
train_batch_size = 2**10

# Use double precision
torch.set_default_dtype(torch.float64)
device = torch.device('cpu')


### Test Data Simulation

In [ ]:
def simulate_forward_process_misspecification(batch_size, dividend_true):
    x = torch.zeros(batch_size, n_time_steps + 1, d, device=device)
    x[:, 0, :] = x_0
    dw = torch.zeros(batch_size, n_time_steps, d, device=device)
    for t in range(n_time_steps):
        dw[:, t, :] = np.sqrt(dt) * torch.randn(batch_size, d, device=device)
        x[:, t + 1, :] = x[:, t, :] * torch.exp(
            (r - dividend_true - 0.5 * sigma ** 2) * dt + sigma * (dw[:, t, :]  )
        )
    return x

### Loading Reference Values

In [ ]:
def load_reference_values():
    data = {
        'dividend_true': [ 0.0, 0.05, 0.1, 0.15, 0.2, 0.25],
        'reference_values': [4.953894, 4.408844, 3.989771, 3.633545,
                         3.323930, 3.052058],
    }
    return pd.DataFrame(data)

dividend_true_list = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]
reference_df = load_reference_values()


### Find Models

In [ ]:
model_dir = "call_trained_models_robust_PI"
def find_trained_models(model_dir = model_dir):
    model_dir = model_dir
    model_files = list(Path(model_dir).glob("*.pt"))
    models = []
    for model_file in model_files:
        model_name = model_file.stem
        try:
            parts = model_name.split('_')
            epsilon = float(parts[2])
            lambda_temp = float(parts[4])
            models.append({
                'model_name': model_name,
                'epsilon': epsilon,
                'lambda_temp': lambda_temp,
            })
        except (IndexError, ValueError) as e:
            print(f"Skipping {model_name}: {e}")
    return models

In [ ]:
print("Finding trained models...")
trained_models = find_trained_models()
if not trained_models:
    print("No trained models found in targeted directory.")
    import sys
    sys.exit(1)
print(f"Found {len(trained_models)} trained models.")


### Robustness Evaluation under $\lambda = 1 $ (one can adjust $ \lambda $)

In [ ]:
lambda_temp = 1  # temperature parameter

eval_params = []
for model in trained_models:
    epsilon = model['epsilon']
    lambda_value = model['lambda_temp']
    model_name = model['model_name']
    if lambda_value == lambda_temp:
        eval_params.append((epsilon, model_name))

print(f"Starting evaluation of {(len(eval_params))} models...")
start_time = time.time()

results = []
for i, dividend_true in enumerate(dividend_true_list):
    ref_row = reference_df[reference_df['dividend_true'].apply(lambda x: abs(x - dividend_true) < 1e-5)]
    reference_value = ref_row.iloc[0]['reference_values']

    x_test = simulate_forward_process_misspecification(test_size, dividend_true)
    for j, (epsilon, model_name) in enumerate(eval_params):
        PE_solver = PE(
            d=d,
            total_time=total_time,
            n_time_steps=n_time_steps,
            K=K,
            r=r,
            dividend = dividend_true,
            sigma=sigma,
            strike=strike,
            x_0=x_0,
            lambda_temp=lambda_temp,
            epsilon = epsilon,
            hidden_layers=2,
            hidden_dim=21,
            test_size = test_size
        )

        if PE_solver.load_model(model_dir, model_name):
            start_time = time.time()

            expected_reward, er_std = PE_solver.evaluate_expected_reward(x_test)
            evaluation_time = time.time() - start_time

            relative_error = abs(expected_reward - reference_value) / reference_value
            print('---------')
            print(f"Completed evaluation: epsilon={epsilon}, dividend_true={dividend_true:.4f}")
            print(f"  Expected Reward: {expected_reward:.6f} ± {er_std:.6f}")
            print(f"  RE: {relative_error:.6f}")
            print(f"  Evaluation time: {evaluation_time:.2f}s")

            result = {
                'epsilon': epsilon,
                'dividend_true': dividend_true,
                'reference_value': reference_value,
                'expected_reward': expected_reward,
                'er_std': er_std,
                'relative_error': relative_error,
                'evaluation_time': evaluation_time,
                'success': True
            }
            results.append(result)

total_time = time.time() - start_time


In [ ]:
successful_eval = [r for r in results if r.get('success', False)]
print(f"Evaluation completed in {total_time:.2f} seconds.")

results_df = pd.DataFrame(successful_eval)
results_df.to_csv(f'call_robustness_results_lambda_{lambda_temp}_penalty_{K}.csv', index=False)


#### Plot Robustness Performance

In [ ]:
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

OUTPUT_DIR = 'call_robustness_performance'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created directory: {OUTPUT_DIR}")

epsilon_colors = [
'#1f77b4',  # Blue
'#2ca02c',   # Green
'#ff7f0e',   # Orange
'#27d6b6',  # Cyan
]

eps_list = [0.0, 0.1, 0.2, 0.3]

def add_training_r_line(ax):
    ax.axvline(x=dividend_train, color='red', linestyle='--', linewidth=3, alpha=0.6,
                   zorder=10, label=f'Training Dividend Rate ({dividend_train})')

def plot_relative_errors_combined(results_df):
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    offline_lambda = results_df

    for i, eps in enumerate(eps_list):
        eps_data = offline_lambda[offline_lambda['epsilon'] == eps].sort_values('dividend_true')
        if len(eps_data) > 0:
            max_num = 6
            ax.plot(eps_data['dividend_true'][:max_num], eps_data['relative_error'][:max_num] * 100,
                    'o-', color=epsilon_colors[i], linewidth=5, markersize=8,
                    label=f'ε={eps}', alpha=0.6)

    add_training_r_line(ax)
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.6, linewidth=3 , label = '1% reference line')
    ax.set_xlabel('True Dividend Rate', fontsize=37)
    ax.set_ylabel('Relative Error (%)', fontsize=37)
    ax.set_title(f'λ = {lambda_temp}', fontsize=37)
    ax.legend(loc='upper right', fontsize=29)
    ax.grid(True, alpha=0.3)

    ax.set_ylim(0,10)
    ax.tick_params(axis='x', labelsize=35)
    ax.tick_params(axis='y', labelsize=35)

    plt.tight_layout()

    filename = os.path.join(OUTPUT_DIR, f'call_RE_lambda_{lambda_temp}.pdf')
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    print(f"Saved: {filename}")
    plt.show()
    plt.close()

plot_relative_errors_combined(results_df)
print(f"RE Plot has been saved to '{OUTPUT_DIR}' directory")
